# Romberg ML Model - Data Preprocessing
## 1. Starter Data
- num of (unique) samples with eyes open: 8
  
  num of non-unique samples: 2
- num of (unique) samples with eyes closed: 8

  num of non-unique samples: 2

so the dataset created will have ten rows

In [20]:
!rm -rf sophia-romberg-data

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 1a. Data Collection:
TODO: Set up script that pulls all new CSV files from a repo/set of repos

Data format:
(https://drive.google.com/drive/folders/1GeIZo_ZzwndkKTcrNOfdFyAkFWzRBl_h?usp=sharing)

/ data

      / 0
          / session_0
              eyes_open.csv
              eyes_closed.csv
              Metadata.csv
          / session_1
              eyes_open.csv
              eyes_closed.csv
              Metadata.csv
          ...
          / session_n
      
      / 1
          / session_0
              eyes_open.csv
              eyes_closed.csv
              Metadata.csv
      ...
      / n


The actual CSV files **should all follow the same naming convention of `Metadata.csv`, `eyes_open.csv` and `eyes_closed.csv` respectively**. Also please title folders accordingly to make things easier.

Repeating multiple sessions with the same subject will not lead to a different label (as it is assumed that a person either always fails or passes the test). But it will give us more training data (rows in the dataset).

This allows the script to just look into each folder in the directory, go through the session folders, and pull the correct CSV files.

## 2. Preprocessing Steps
- set label as `closed` or `open` for each data point depending on the file name. this will be the label for the rows.
- add unique `session_id` corresponding to **each recording session.**
- add a column for `subject_id` for who the data was recorded from (don't use actual names for privacy reasons, instead use integer naming like 0, 1, 2, 3, etc)
- add columns for computed duration of eyes open and closed
- add column for the label, ie 'stable' or 'impaired' -- will need to one-hot encode this later
- compute some stats from the accelerometer stats for eyes open/closed

Proposed data columns:
- `session_id`
- `subject_id`
- `label`
- `eyes_open_duration`
- `eyes_closed_duration`
- `rms_open`, `rms_closed`
- `std_mag_open`, `std_mag_closed`
- `romberg_ratio_rms`
- `romberg_ratio_std`

Note: some of these features can be removed later but it's better to start with more features in the beginning. I used this chatGPT conversation for figuring out some features to use: https://chatgpt.com/share/69bed09c-0c84-8012-a8da-42470ea86154

An interesting article on the use of the Romberg ratio: https://www.sciencedirect.com/science/article/pii/S0966636214007905

## 3. Training
- train/test/valid split: 70/20/10
- IMPORTANT: split data by subject, not session

In [2]:
!git clone https://github.com/A2AppRom/sophia-romberg-data

Cloning into 'sophia-romberg-data'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 72 (delta 9), reused 72 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 4.59 MiB | 13.40 MiB/s, done.
Resolving deltas: 100% (9/9), done.


In [2]:
import os
import pandas as pd
import numpy as np

In [3]:
ROOT = '/content/drive/MyDrive/cs410/data'
subjects = os.listdir(ROOT)

In [4]:
cols = ['subject_id', 'session_id', 'label', 'duration', 'rms', 'std_mag',
        'romberg_ratio_rms', 'romberg_ratio_std']

In [25]:
def get_label(filename):
  if 'eyes_open' in filename:
    return 'open'
  if 'eyes_closed' in filename:
    return 'closed'
  else:
    return 'error'

In [5]:
df = pd.DataFrame(columns=cols)

In [6]:
!rm -rf .DS_Store

In [7]:
for i in range (len(subjects)):
  sub_path = os.path.join(ROOT, subjects[i])
  sessions = os.listdir(sub_path)

  for j in range(len(sessions)):
    session = os.path.join(sub_path, sessions[j])

    open = pd.read_csv(os.path.join(session, 'eyes_open.csv'))
    closed = pd.read_csv(os.path.join(session, 'eyes_closed.csv'))

    open_time_diff = open['time'].iloc[-1] - open['time'].iloc[0]
    closed_time_diff = closed['time'].iloc[-1] - closed['time'].iloc[0]

    open['mag'] = np.sqrt(open['x']**2 + open['y']**2 + open['z']**2)
    closed['mag'] = np.sqrt(closed['x']**2 + closed['y']**2 + closed['z']**2)

    RMS_open = np.sqrt(np.mean(open['mag']**2))
    RMS_closed = np.sqrt(np.mean(closed['mag']**2))

    std_mag_open = np.std(open['mag'])
    std_mag_closed = np.std(closed['mag'])

    romberg_ratio_rms = RMS_closed / RMS_open
    romberg_ratio_std = std_mag_closed / std_mag_open

    # add data with eyes open label
    df_temp_open = pd.DataFrame(columns=cols)
    df_temp_open.loc[len(df_temp_open)] = {"label": "open",
                                 "session_id": sessions[j][8:],
                                 "subject_id": subjects[i][8:],
                                 "duration": open_time_diff,
                                 "rms": RMS_open,"std_mag": std_mag_open,
                                 "romberg_ratio_rms": romberg_ratio_rms,
                                 "romberg_ratio_std": romberg_ratio_std}

    df = pd.concat([df, df_temp_open], ignore_index=True)

    # add data with eyes closed label
    df_temp_closed = pd.DataFrame(columns=cols)
    df_temp_closed.loc[len(df_temp_closed)] = {"label": "closed", "session_id": sessions[j][8:],
                                 "subject_id": subjects[i][8:],
                                 "duration": closed_time_diff,
                                 "rms": RMS_closed,"std_mag": std_mag_closed,
                                 "romberg_ratio_rms": romberg_ratio_rms,
                                 "romberg_ratio_std": romberg_ratio_std}

    df = pd.concat([df, df_temp_closed], ignore_index=True)
df


/tmp/ipykernel_5849/1545267323.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, df_temp_open], ignore_index=True)


,subject_id,session_id,label,duration,rms,std_mag,romberg_ratio_rms,romberg_ratio_std
0,1,0,open,58753421600,0.584403,0.556785,1.101173,1.062279
1,1,0,closed,45567372600,0.643529,0.591461,1.101173,1.062279
2,2,0,open,42846806800,0.366372,0.328285,1.124497,1.145674
3,2,0,closed,43068919300,0.411984,0.376108,1.124497,1.145674
4,3,0,open,42798669300,0.383673,0.350194,1.040417,1.029774
5,3,0,closed,42738772700,0.399179,0.360620,1.040417,1.029774
6,4,0,open,32043969000,0.736683,0.621835,0.510672,0.496044
7,4,0,closed,34405365500,0.376203,0.308457,0.510672,0.496044
8,0,0,open,45564259500,0.400963,0.342887,1.479420,1.619363
9,0,0,closed,45324047600,0.593192,0.555259,1.479420,1.619363


In [8]:
df.to_csv("labeled_data_with_features_NEW.csv")